In [9]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import pickle
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent 
DATA_PATH = BASE_DIR / "data" / "cleaned_resumes.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (2483, 9)
Columns: ['ID', 'Resume_str', 'Category', 'cleaned_resume', 'target', 'years_experience', 'has_degree', 'skill_keywords', 'embed_resume']


In [10]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("./models/all-MiniLM-L6-v2")

def embed_text(text):
    vector = embedder.encode(text, convert_to_numpy=True)
    vector = vector / np.linalg.norm(vector)  # normalize for cosine via inner product
    return vector.reshape(1, -1).astype('float32')

# Sanity test BEFORE building full index
# Pick two IT resumes and one non-IT resume
it_rows  = df[df['Category'] == 'INFORMATION-TECHNOLOGY'].iloc[:2]
chef_row = df[df['Category'] == 'CHEF'].iloc[0]

v_it1  = embed_text(it_rows.iloc[0]['embed_resume'])
v_it2  = embed_text(it_rows.iloc[1]['embed_resume'])
v_chef = embed_text(chef_row['embed_resume'])

sim_same = np.dot(v_it1, v_it2.T)[0][0]
sim_diff = np.dot(v_it1, v_chef.T)[0][0]

print(f"IT vs IT similarity:   {sim_same:.3f}") 
print(f"IT vs CHEF similarity: {sim_diff:.3f}") 

IT vs IT similarity:   0.515
IT vs CHEF similarity: 0.228


In [ ]:
import faiss
import pickle

texts = df['embed_resume'].tolist()

print(f"Embedding {len(texts)} resumes...")
vectors = embedder.encode(texts,batch_size=64,show_progress_bar=True,convert_to_numpy=True,normalize_embeddings=True)
vectors = vectors.astype('float32')
print(f"Vectors shape: {vectors.shape}") 

# Build index
index = faiss.IndexFlatIP(384)
index.add(vectors)
print(f"Index total vectors: {index.ntotal}") 
TECH_CATEGORIES = ['INFORMATION-TECHNOLOGY', 'ENGINEERING']
# Build metadata
metadata = []
for i, row in df.iterrows():
    metadata.append({
        'candidate_id': int(i),
        'category':     row['Category'],
        'is_tech':      row['Category'] in TECH_CATEGORIES,
        'text':         row['embed_resume'][:1000],
        'fit_score':    float(row.get('target', 0)),
    })

print(f"Metadata entries: {len(metadata)}")

Embedding 2483 resumes...


Batches: 100%|██████████| 39/39 [04:00<00:00,  6.16s/it]


Vectors shape: (2483, 384)
Index total vectors: 2483
Metadata entries: 2483


In [14]:
def retrieve(query, k=5):
    q_vec = embed_text(query)
    distances, indices = index.search(q_vec, k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        r = metadata[idx].copy()
        r['similarity_score'] = round(float(dist), 4)
        results.append(r)
    return results

print("=== Python ML developer ===")
for r in retrieve("Python developer machine learning tensorflow"):
    print(f"  {r['category']:<30} {r['similarity_score']:.3f}")

print("\n=== Chef kitchen ===")
for r in retrieve("head chef kitchen restaurant culinary"):
    print(f"  {r['category']:<30} {r['similarity_score']:.3f}")

print("\n=== Doctor hospital ===")
for r in retrieve("doctor physician hospital clinical medicine"):
    print(f"  {r['category']:<30} {r['similarity_score']:.3f}")

=== Python ML developer ===
  ENGINEERING                    0.403
  ENGINEERING                    0.333
  ENGINEERING                    0.273
  ENGINEERING                    0.273
  AUTOMOBILE                     0.272

=== Chef kitchen ===
  CHEF                           0.746
  CHEF                           0.721
  CHEF                           0.713
  CHEF                           0.703
  CHEF                           0.703

=== Doctor hospital ===
  HEALTHCARE                     0.562
  HEALTHCARE                     0.537
  HEALTHCARE                     0.508
  HEALTHCARE                     0.507
  INFORMATION-TECHNOLOGY         0.494


In [16]:
import os
from pathlib import Path
BASE_DIR    = Path.cwd().parent
MODELS_DIR  = BASE_DIR / "models"

faiss_path    = MODELS_DIR / "resume_index.faiss"
metadata_path = MODELS_DIR / "metadata.pkl"

faiss.write_index(index, str(faiss_path))
with open(metadata_path, 'wb') as f:
    pickle.dump(metadata, f)

print(f"resume_index.faiss: {os.path.getsize(str(faiss_path))/1024:.1f} KB")
print(f"metadata.pkl:       {os.path.getsize(str(metadata_path))/1024:.1f} KB")

resume_index.faiss: 3724.5 KB
metadata.pkl:       2515.5 KB
